In [21]:
import ollama  
import time 

all_models = [m.model for m in ollama.list().models]


print(f"These are the available models: {(all_models)}")

These are the available models: ['qwen2.5:7b', 'nimble:latest', 'deepseek-r1:latest', 'llama3:latest']


In [22]:
prompt = "What ist the capital of France?"

In [32]:
def ollama_chat(prompt, model):
    stream = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        stream=True,
    )

    output = ""
    for chunk in stream:
        #print(chunk.message.content, end="", flush=True)
        output += chunk.message.content
    return output

In [35]:
output = ollama_chat(prompt, all_models[0])
print(f"{all_models[0]} :  {output}")

qwen2.5:7b :  The capital of France is Paris.


In [37]:
import os, re, httpx
from pydantic_ai import Agent, ModelRetry
from dotenv import load_dotenv
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


load_dotenv()  # vor allem anderen, was die Keys braucht

DIP = "https://search.dip.bundestag.de/api/v1"
KEY = os.environ["DIP_API_KEY"]

model = OpenAIChatModel(
    "qwen2.5:7b",
    provider=OpenAIProvider(base_url="http://localhost:11434/v1", api_key="ollama"),
)
agent = Agent(model)

@agent.tool_plain
def suche_vorgaenge(titel: str, datum_von: str, datum_bis: str) -> str:
    """Sucht Gesetzgebungsvorgänge im DIP des Bundestags nach Stichwort im Titel.

    Args:
        titel: Stichwort, z.B. "Mindestlohn".
        datum_von: Startdatum, Format YYYY-MM-DD.
        datum_bis: Enddatum, Format YYYY-MM-DD.
    """
    for name, wert in (("datum_von", datum_von), ("datum_bis", datum_bis)):
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", wert):
            raise ModelRetry(f"{name} muss YYYY-MM-DD sein, z.B. 2025-01-31")

    r = httpx.get(f"{DIP}/vorgang", timeout=15, headers={"Authorization": f"ApiKey {KEY}"},
                  params={"f.titel": titel, "f.datum.start": datum_von, "f.datum.end": datum_bis})
    if r.status_code == 401:
        raise ModelRetry("API-Key ungültig, Suche nicht möglich.")
    r.raise_for_status()

    docs = r.json()["documents"]
    if not docs:
        raise ModelRetry(f"Keine Treffer für '{titel}'. Versuch ein kürzeres Stichwort.")
    return "\n".join(f"{d['datum']} | {d['vorgangstyp']} | {d['titel']}" for d in docs[:10])



result = await agent.run("Was lief 2025 zum Mindestlohn?")
print(result.output)

Es gab Kleine Anträge zum Thema Mindestlohn im Deutschen Bundestag, die zum Teil 2025 referentiert wurden. Hier ist eine Zusammenfassung:

- Eine Kleine Anfrage am 19. Januar 2026 über Rechtssichere Festsetzung des gesetzlichen Mindestlohns und Prüfungen durch die Bundesregierung.
- Mehrere Anfragen am 1. September 2025 und 29. August 2025 betrafen den Mindestlohnbetrug und die Kontrollen zur Einhaltung des gesetzlichen Mindestlohns in verschiedenen Bundesländern und bei der Einhaltung des allgemeinen gesetzlichen Mindestlohns in Deutschland insgesamt.

Diese Daten deuten darauf hin, dass das Thema Mindestlöhne zu diesem Zeitpunkt sehr aktuell war und in verschiedenen Kontexten besprochen wurde.
